# Chapter 11 — Problem Set 1: Solutions

Runnable solutions with explanations. Alternative approaches are noted where relevant.

---
*Generated by Berta AI*

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'scripts'))
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

## 1. Scaled Dot-Product Attention — Solution

In [ ]:
def stable_softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def my_sdp_attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    weights = stable_softmax(scores, axis=-1)
    return weights @ V, weights

# Verify against the chapter implementation
from transformer_utils import scaled_dot_product_attention
Q, K, V = np.random.randn(6, 8), np.random.randn(6, 8), np.random.randn(6, 8)
out_mine, w_mine = my_sdp_attention(Q, K, V)
out_ref, w_ref = scaled_dot_product_attention(Q, K, V)
print("Max abs diff (output):", np.max(np.abs(out_mine - out_ref)))
print("Max abs diff (weights):", np.max(np.abs(w_mine - w_ref)))

## 2. Sinusoidal Positional Encoding — Solution

In [ ]:
def my_positional_encoding(seq_len, d_model):
    pos = np.arange(seq_len)[:, None]
    i = np.arange(d_model)[None, :]
    rates = 1.0 / np.power(10000.0, (2 * (i // 2)) / d_model)
    angles = pos * rates
    pe = np.zeros((seq_len, d_model))
    pe[:, 0::2] = np.sin(angles[:, 0::2])
    pe[:, 1::2] = np.cos(angles[:, 1::2])
    return pe

pe = my_positional_encoding(64, 32)
print("PE shape:", pe.shape)

fig, ax = plt.subplots(figsize=(8, 3))
im = ax.imshow(pe, aspect='auto', cmap='RdBu')
ax.set_xlabel('dim'); ax.set_ylabel('position'); ax.set_title('Positional encoding')
fig.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

# Column 0 is sin(pos)
print("Column 0 first 5 values:", pe[:5, 0].round(3))

## 3. Attention Heatmap — Solution

In [ ]:
tokens = ['the', 'cat', 'sat', 'on', 'the', 'mat']
d = 8
rng = np.random.default_rng(0)
emb = rng.standard_normal((len(tokens), d))
out, weights = my_sdp_attention(emb, emb, emb)

fig, ax = plt.subplots(figsize=(4, 3.5))
im = ax.imshow(weights, cmap='viridis')
ax.set_xticks(range(len(tokens))); ax.set_yticks(range(len(tokens)))
ax.set_xticklabels(tokens, rotation=45, ha='right'); ax.set_yticklabels(tokens)
ax.set_xlabel('Key'); ax.set_ylabel('Query'); ax.set_title('Self-attention')
fig.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

print('Note: with random embeddings, attention is roughly uniform — real models learn structure.')

## 4. Tokenize Text & Reason About BPE — Solution

In [ ]:
text = "unbelievably preprocessed"
try:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained('distilbert-base-uncased')
    pieces = tok.tokenize(text)
    print('Pieces:', pieces)
    print('Continuation pieces start with ##:', [p for p in pieces if p.startswith('##')])
except Exception as e:
    print(f'transformers not installed ({e}); manual demo.')
    pieces = ['un', '##believ', '##ably', 'pre', '##process', '##ed']
    print('Pieces (manual):', pieces)
    print('Continuation pieces:', [p for p in pieces if p.startswith('##')])

print('\nWhy no OOV: BPE merges are learned over bytes/characters, so any string '
      'can fall back to a sequence of single-character tokens.')

## 5. Multi-Head Attention Shape Check — Solution

In [ ]:
from transformer_utils import MultiHeadAttention

d_model, num_heads, seq_len, batch = 64, 8, 10, 2
assert d_model % num_heads == 0
mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads, seed=0)
x = np.random.randn(batch, seq_len, d_model)
y = mha(x)

print('input x      :', x.shape)
print('output y     :', y.shape)
print('attn weights :', mha.last_attn_weights.shape)  # (batch, heads, seq, seq)
print('d_head       :', mha.d_head, '(d_model / num_heads)')
print('Sanity:', mha.d_head * num_heads == d_model)

## 6. Encoder / Decoder / Encoder–Decoder — Solution

| Task | Family | Why |
|------|--------|-----|
| Sentiment classification | Encoder | Need a single vector, not generation |
| Story continuation | Decoder | Autoregressive generation from a prefix |
| English → French translation | Encoder–Decoder | Bidirectional encoding of source + autoregressive target |
| Document summarisation | Encoder–Decoder (or decoder w/ long context) | Read all → write summary |
| Named-entity extraction | Encoder | Per-token tagging benefits from bidirectional context |